In [ ]:
import gradio as gr
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from skimage.color import rgb2lab
from models import UNetGenerator, lab_to_rgb_numpy

def load_model():
    model = UNetGenerator()
    model.load_state_dict(torch.load("best_model7.pth", map_location="cpu"))
    model.eval()
    return model

model = load_model()


def preprocess(image: Image.Image):
    # Convert to RGB and resize
    transform = transforms.Resize((256, 256))
    
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    image = transform(image)
    
    # Convert to numpy array and normalize to [0, 1]
    img_np = np.array(image, dtype=np.float32) / 255.0
    
    # Convert RGB to LAB
    lab = rgb2lab(img_np).astype(np.float32)
    
    # Extract and normalize L channel: [0, 100] → [-1, 1]
    L = lab[:, :, 0] / 50.0 - 1.0
    
    # Convert to tensor and add batch dimension
    L_tensor = torch.from_numpy(L).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
    
    return L_tensor


def postprocess(output_tensor, L_tensor, original_size):
    # output_tensor shape: (1, 2, H, W) — ab output from model
    # L_tensor shape: (1, 1, H, W) — L channel from input
    
    # Squeeze batch dimension
    ab = output_tensor.squeeze(0)  # (2, H, W)
    L = L_tensor.squeeze(0)        # (1, H, W)
    
    # Convert LAB back to RGB using lab_to_rgb_numpy
    rgb = lab_to_rgb_numpy(L, ab)
    
    result = Image.fromarray((rgb * 255).astype(np.uint8))
    result = result.resize(original_size, Image.LANCZOS)  # restore original size
    return result


def colorize(image: Image.Image):
    original_size = image.size  # (W, H)
    
    input_tensor = preprocess(image)
    
    with torch.no_grad():
        output_tensor = model(input_tensor)
    
    colorized = postprocess(output_tensor, input_tensor, original_size)
    return colorized


demo = gr.Interface(
    fn=colorize,
    inputs=gr.Image(
        type="pil",
        label="Input — grayscale or color image"
    ),
    outputs=gr.Image(
        type="pil",
        label="Colorized Output"
    ),
    title="🎨 Image Colorization",
    description="Upload a grayscale (or color) image and the model will colorize it.",
    examples=[],          # optionally add: [["example1.jpg"], ["example2.jpg"]]
    flagging_mode="never" # disables the Flag button
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://6d32cf6c7c859cbf7e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/home/user/Documents/deep-learning/labs/lab3/models.py:132: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 96 negative Z values that have been clipped to zero
  return np.clip(lab2rgb(lab.astype(np.float64)), 0, 1)
/home/user/Documents/deep-learning/labs/lab3/models.py:132: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 82 negative Z values that have been clipped to zero
  return np.clip(lab2rgb(lab.astype(np.float64)), 0, 1)
/home/user/Documents/deep-learning/labs/lab3/models.py:132: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 82 negative Z values that have been clipped to zero
  return np.clip(lab2rgb(lab.astype(np.float64)), 0, 1)
